**Import libraries for data exploration**

In [ ]:
%pip install pandas
%pip install regex

In [149]:
import pandas as pd
import os
import re
import datetime
import 

I want to combine all the raw order forms into one dataframe for data exploration. First, I'm going to create dataframes for each order form and combine them into one dataframe, creating a unique column for each order form using the first timestamp of each order form.

In [ ]:
working_dir = os.getcwd()
order_forms_raw_dir = os.path.join(working_dir, 'order_forms_raw')
order_forms_processed_dir = os.path.join(working_dir, 'order_forms_processed')

df = None

for file in os.listdir(order_forms_raw_dir):
    if file.endswith('.csv'):
        # read the current CSV file
        current_df = pd.read_csv(f'order_forms_raw/{file}')
        
        # get first timestamp from this file to use as identifier
        form_id = current_df['Timestamp'].iloc[0]
        
        # add column identifying which form this came from
        current_df['Form_ID'] = form_id
        
        # add to main dataframe
        if df is None:
            df = current_df
        else:
            df = pd.concat([df, current_df])

if not os.path.exists(order_forms_processed_dir):
    os.makedirs(order_forms_processed_dir)

df.to_csv(os.path.join(order_forms_processed_dir, 'combined_order_forms.csv'), index=False)
print(df.info())


Next, I want to remove rows with no timestamps and rows with duplicate timestamps.

In [ ]:
df = df[df['Timestamp'].notna()] # remove rows with no timestamps
df = df.drop_duplicates(subset=['Timestamp']) # remove duplicate timestamps

print(df.info())

Next, I'm going to remove columns "Unnamed: 8" and "Unnamed: 11"

In [ ]:
df = df.drop(columns=["Unnamed: 8","Unnamed: 11"]) # remove empty columns

df.to_csv(os.path.join(order_forms_processed_dir, 'combined_order_forms.csv'), index=False)
print(df.info())

Since we have a very large sample of orders, I'm just going to remove rows that have incomplete names, started, and complete data since they make up less than 1% of the data and we aren't doing anything super rigorous here.

In [ ]:
df = df[(df["NAME"].notna() & df["Started"].notna() & df["Completed"].notna())]

df.info()

It's looking good! Next, I want to look at the rows where tea data is missing.

In [ ]:
blank_tea = df[df["TEA"].isna()]

blank_tea.head(20)

It looks like the tea data is missing for some test runs, really custom orders (I'm looking at you Cindy), or fuckit bucket orders, which I don't want to train our model on. While there's some legitimate orders, I'm going to remove all rows with missing tea data to avoid issues down the road.

In [ ]:

df = df[df["TEA"].notna()]

df.to_csv(os.path.join(order_forms_processed_dir, "order_forms_pre_sort.csv"), index=False)
df.info()

Finally, I want to standardize the timestamps to be in the format of YYYY-MM-DD HH:MM:SS, convert them to datetime objects, and then sort the dataframe by timestamp. Lets check if any timestamps are in the wrong format.

In [ ]:
# regex pattern to match example timestamp "2/9/2023 19:11:27"
timestamp_pattern = r'\d{1,2}/\d{1,2}/\d{4}\s\d{2}:\d{2}:\d{2}'

# find rows where timestamp doesn't match the pattern
wrong_format = df[~df["Timestamp"].str.match(timestamp_pattern)]

wrong_format.head()

Yay! All our timestamps are in the same format (which makes sense since they were all created by google forms). I'm going to convert them to datetime objects, sort the dataframe by timestamp, and save the dataframe to a new csv.

In [ ]:
df["Timestamp"] = pd.to_datetime(df["Timestamp"], format='%m/%d/%Y %H:%M:%S')
df.sort_values(by="Timestamp", inplace=True)

df.to_csv(os.path.join(order_forms_processed_dir, "order_forms_pre_sort.csv")) # saves with index